In [2]:
# ============================================================
# 10. IMPROVE RETRIEVAL AND PROMPTS
# ============================================================

import pandas as pd

# ============================================================
# STEP 1: LOAD YOUR ACTUAL DATASET
# ============================================================

df = pd.read_csv("/content/smartagri_crop_yield_dataset.csv")

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

print("\nAvailable Crops:")
print(df["Crop"].unique())


# ============================================================
# STEP 2: BUILD AGRICULTURAL KNOWLEDGE BASE
# ============================================================

knowledge_base = df.groupby("Crop").agg({

    "Soil_Type": lambda x: x.mode()[0],

    "Soil_pH": "mean",

    "Nitrogen": "mean",

    "Phosphorus": "mean",

    "Potassium": "mean",

    "Temperature": "mean",

    "Rainfall": "mean",

    "Humidity": "mean",

    "Yield": "mean"

}).reset_index()


print("\n================================================")
print("        AGRICULTURAL KNOWLEDGE BASE")
print("================================================")

print(
    knowledge_base.round(2).to_string(index=False)
)


# ============================================================
# STEP 3: IMPROVED RETRIEVAL FUNCTION
# ============================================================

def retrieve_crop_information(crop):

    result = knowledge_base[
        knowledge_base["Crop"].str.lower()
        == crop.lower()
    ]

    if len(result) == 0:

        return None

    return result.iloc[0]


# ============================================================
# STEP 4: CREATE IMPROVED PROMPT
# ============================================================

def create_farmer_prompt(crop, farmer_question):

    information = retrieve_crop_information(crop)

    # Crop not available
    if information is None:

        return "Crop information is not available in the dataset."


    prompt = f"""
You are a smart agriculture advisory assistant.

Use only the agricultural information provided below.

CROP INFORMATION
----------------
Crop: {information['Crop']}
Soil Type: {information['Soil_Type']}
Average Soil pH: {information['Soil_pH']:.2f}
Average Nitrogen: {information['Nitrogen']:.2f}
Average Phosphorus: {information['Phosphorus']:.2f}
Average Potassium: {information['Potassium']:.2f}
Average Temperature: {information['Temperature']:.2f}
Average Rainfall: {information['Rainfall']:.2f}
Average Humidity: {information['Humidity']:.2f}
Average Yield: {information['Yield']:.2f}

FARMER QUESTION
---------------
{farmer_question}

INSTRUCTIONS
------------
1. Answer the farmer's question clearly.
2. Use the crop information provided above.
3. Give simple soil advice.
4. Give water or irrigation advice when relevant.
5. Give nutrient advice when relevant.
6. Mention possible crop risks when relevant.
7. Give one simple action for the farmer.
8. Do not invent information that is not available.
9. If the information is insufficient, clearly say so.
"""

    return prompt


# ============================================================
# STEP 5: TEST FARMER QUESTIONS
# ============================================================

test_questions = [

    (
        "Rice",
        "What soil condition is suitable for rice?"
    ),

    (
        "Wheat",
        "What is the rainfall condition for wheat?"
    ),

    (
        "Maize",
        "What nutrient information is available for maize?"
    ),

    (
        "Cotton",
        "What humidity condition should I monitor for cotton?"
    ),

    (
        "Tomato",
        "What is the soil pH information for tomato?"
    ),

    (
        "Groundnut",
        "What is the expected yield information for groundnut?"
    )
]


# ============================================================
# STEP 6: GENERATE IMPROVED PROMPTS
# ============================================================

print("\n================================================")
print("       IMPROVED FARMER PROMPTS")
print("================================================")

for i, (crop, question) in enumerate(
    test_questions, 1
):

    print("\n--------------------------------------------")
    print("Test Case:", i)
    print("Crop:", crop)
    print("Farmer Question:", question)
    print("--------------------------------------------")

    prompt = create_farmer_prompt(
        crop,
        question
    )

    print(prompt)


# ============================================================
# STEP 7: TEST UNSUPPORTED CROP
# ============================================================

print("\n================================================")
print("       UNSUPPORTED CROP TEST")
print("================================================")

unsupported_prompt = create_farmer_prompt(
    "Banana",
    "What soil is suitable for banana?"
)

print(unsupported_prompt)


# ============================================================
# STEP 8: RETRIEVAL SUMMARY
# ============================================================

successful_retrievals = 0
failed_retrievals = 0

for crop, question in test_questions:

    information = retrieve_crop_information(crop)

    if information is not None:
        successful_retrievals += 1
    else:
        failed_retrievals += 1


total_tests = len(test_questions)

retrieval_percentage = (
    successful_retrievals / total_tests
) * 100


# ============================================================
# STEP 9: FINAL SUMMARY
# ============================================================

print("\n================================================")
print("          RETRIEVAL SUMMARY")
print("================================================")

print(
    "Total retrieval tests:",
    total_tests
)

print(
    "Successful retrievals:",
    successful_retrievals
)

print(
    "Failed retrievals:",
    failed_retrievals
)

print(
    "Retrieval success:",
    round(retrieval_percentage, 2),
    "%"
)

print("\nImproved retrieval and prompt generation completed!")

Dataset loaded successfully!
Dataset Shape: (600, 10)

Available Crops:
['Millets' 'Sugarcane' 'Maize' 'Cotton' 'Rice' 'Wheat' 'Groundnut']

        AGRICULTURAL KNOWLEDGE BASE
     Crop Soil_Type  Soil_pH  Nitrogen  Phosphorus  Potassium  Temperature  Rainfall  Humidity  Yield
   Cotton     Loamy     6.83     63.97       28.20      41.45        27.19    661.18     59.43   2.09
Groundnut     Loamy     6.34     48.75       30.42      38.23        26.11    605.93     65.42   1.80
    Maize  Alluvial     6.24     76.09       36.63      39.57        24.47    687.87     68.48   3.64
  Millets     Loamy     6.13     41.65       23.46      33.63        26.93    444.10     57.99   1.45
     Rice     Loamy     6.35     81.41       38.43      41.82        26.76   1225.18     80.87   4.00
Sugarcane     Loamy     7.00     94.72       41.95      50.02        27.77   1439.81     78.58  66.28
    Wheat     Black     6.70     69.11       32.66      37.68        21.78    536.64     61.78   3.32

      